# Riset Model Pembaca Nota — SmartSplit Bill

Notebook ini adalah **Step 1** assignment: mencoba beberapa model untuk mengekstrak data
dari foto nota, lalu membandingkan **akurasi, reliability, dan kecepatan inference**.

Data yang harus terbaca (requirement D):
1. tiap item: nama, jumlah, harga satuan, total harga item
2. subtotal
3. biaya tambahan (pajak, service charge, diskon, dll.)
4. total bill

## Kandidat model

| # | Model | Jenis | Jalan di | Alasan dipilih |
|---|---|---|---|---|
| 1 | **DeepSeek-V4.1-Flash** (`deepseek-flash`) | VLM proprietary | API (cloud) | Cloud vision model yang mendukung input gambar dan structured output |
| 2 | **Donut** (`naver-clova-ix/donut-base-finetuned-cord-v2`) | OCR-free document understanding, di-*finetune* khusus struk (dataset CORD) | Lokal | Model spesialis struk, kecil (~200M param), tanpa OCR terpisah |
| 3 | **Qwen3-VL-2B-Instruct** (`Qwen/Qwen3-VL-2B-Instruct`) | VLM open-source | Lokal | VLM lokal yang dapat membaca gambar dan mengikuti prompt ekstraksi terstruktur |

Ketiganya **OCR-free** (tidak memakai EasyOCR/PyTesseract): gambar langsung diubah menjadi data terstruktur.
DeepSeek dan Qwen diberi **prompt yang sama** (`modules/readers/prompt.py`) supaya perbandingannya lebih adil.
Donut bekerja dengan pendekatan yang berbeda karena merupakan model document understanding yang sudah di-*fine-tune* pada CORD.

Semua model memakai reader yang sama dengan aplikasi Streamlit (`modules/readers/`), jadi model yang diuji di notebook ini
adalah model yang benar-benar dipakai oleh aplikasi.

## Pertanyaan research

Research ini ingin menjawab empat pertanyaan sederhana:

1. **Akurasi:** model mana yang paling tepat membaca item dan angka pada nota?
2. **Reliability:** seberapa sering output model benar-benar bisa diproses aplikasi?
3. **Kecepatan:** model mana yang paling cepat melakukan inference?
4. **Trade-off:** apakah model yang paling akurat juga paling cocok dipakai pada aplikasi?

## Cara evaluasi

- **Ground truth**: tiap nota diketik manual ke `research/ground_truth/<nama_nota>.json`.
- **Akurasi** (`research/metrics.py`): F1 deteksi item, CER nama item, akurasi jumlah/harga,
  ketepatan subtotal, biaya tambahan, total, dan konsistensi hitungan.
- **Reliability**:
  - `run_success_rate` = persentase seluruh inference yang menghasilkan output yang bisa diparse.
  - `receipt_parse_rate` = persentase nota yang berhasil diparse minimal satu kali.
- **Kecepatan**: waktu load model + waktu inference per nota.
- **Memori**: dicatat sebagai informasi tambahan, tetapi delta RSS tidak selalu cocok untuk membandingkan model lokal di MPS.


## Hardware Acceleration & PyTorch Check
Mendeteksi apakah PyTorch mendeteksi dan menggunakan GPU Macbook (MPS)

In [1]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"MPS built: {torch.backends.mps.is_built()}")

# Menentukan device aktif
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active device: {device}")

PyTorch Version: 2.9.0
MPS available: True
MPS built: True
Active device: mps


In [2]:
# Mengecek ketersediaan Library yg digunakan utk menjalankan project
import PIL
import pandas
import psutil
import streamlit
import transformers

print(f"Transformers: {transformers.__version__}")
print(f"Streamlit: {streamlit.__version__}")
print(f"Pandas: {pandas.__version__}")
print(f"Pillow: {PIL.__version__}")
print(f"psutil: {psutil.__version__}")

Transformers: 4.57.1
Streamlit: 1.64.0
Pandas: 2.3.3
Pillow: 12.3.0
psutil: 7.2.2


In [3]:
import os
import sys
from pathlib import Path

# supaya bisa import package `modules` & `research` dari root project
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import json
import matplotlib.pyplot as plt
import pandas as pd

from modules.readers.prompt import RECEIPT_PROMPT
from research import benchmark
from research.benchmark import (
    benchmark_model,
    environment_info,
    load_dataset,
    save_results,
)

pd.set_option("display.max_colwidth", 60)

# Model lokal dijalankan 3x untuk mendapatkan rata-rata waktu inference.
RUNS = 3

# DeepSeek cukup 1x per nota untuk menghemat penggunaan cloud API.
DEEPSEEK_RUNS = 1

print(json.dumps(environment_info(), indent=2))

{
  "platform": "macOS-26.5.2-arm64-arm-64bit",
  "processor": "arm",
  "python": "3.11.16",
  "cpu_count": 12,
  "ram_gb": 32.0,
  "torch": "2.9.0",
  "cuda": false,
  "mps": true
}


### Interpretasi environment

Eksperimen dijalankan pada **MacBook Pro M2 Max Apple Silicon dengan RAM 32 GB** dan PyTorch mendeteksi **MPS**.
Artinya Donut dan Qwen dapat menjalankan inference menggunakan GPU Apple secara lokal.

DeepSeek berbeda karena inference dilakukan melalui **cloud API**, sehingga angka waktu dan penggunaan resource-nya
tidak sepenuhnya setara dengan model lokal.

## 1. Dataset: foto nota + ground truth

In [4]:
# mengecek jumlah nota
dataset = load_dataset()
print(f"Jumlah nota: {len(dataset)}")

Jumlah nota: 20


In [5]:
# Menampilkan gambar semua Nota yg digunakan
import math

n = len(dataset)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(16, 5 * rows)
)

axes = axes.flatten()

for ax, (name, image, _) in zip(axes, dataset):
    ax.imshow(image)
    ax.set_title(f"{name}\n{image.width}x{image.height}")
    ax.axis("off")

# Sembunyikan subplot yang tidak terpakai
for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

/var/folders/7j/1_v4mdn15nn4ry80ckw0xy9w0000gn/T/ipykernel_11235/1901611225.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
def receipt_table(receipt):
    items = pd.DataFrame([i.model_dump() for i in receipt.items])
    summary = pd.DataFrame(
        [{"name": "SUBTOTAL", "total_price": receipt.subtotal}]
        + [{"name": c.name, "total_price": c.amount} for c in receipt.charges]
        + [{"name": "TOTAL", "total_price": receipt.total}]
    )
    return pd.concat([items, summary], ignore_index=True)

for name, _, gt in dataset:
    print(f"Ground truth {name}")
    display(receipt_table(gt))

Ground truth nota_01


,name,quantity,unit_price,total_price
0,SARI ROTI SW CK,1.0,4500.0,4500.0
1,SUBTOTAL,NaN,NaN,4500.0
2,TOTAL,NaN,NaN,4500.0


Ground truth nota_02


,name,quantity,unit_price,total_price
0,SARI ROTI KRM CK,1.0,3500.0,3500.0
1,OVALTINE 3IN1 12X,1.0,28500.0,28500.0
2,SUBTOTAL,NaN,NaN,32000.0
3,TOTAL,NaN,NaN,32000.0


Ground truth nota_03


...

,name,quantity,unit_price,total_price
0,9PCS CHIC-WINGS,2.0,152273.0,304546.0
1,CHAFEE TA,4.0,1818.0,7272.0
2,SUBTOTAL,NaN,NaN,311818.0
3,P.Rest 10%,NaN,NaN,31182.0
4,TOTAL,NaN,NaN,343000.0


Ground truth nota_19


,name,quantity,unit_price,total_price
0,Pistachio,1.0,12727.0,12727.0
1,Kiwi Breeze,1.0,12727.0,12727.0
2,Air Mineral,1.0,8181.0,8181.0
3,SUBTOTAL,NaN,NaN,33635.0
4,PB1,NaN,NaN,3364.0
5,Pembulatan,NaN,NaN,1.0
6,TOTAL,NaN,NaN,37000.0


Ground truth nota_20


,name,quantity,unit_price,total_price
0,Ayam Bakar Rica,1.0,22210.0,22210.0
1,Paket R,1.0,8000.0,8000.0
2,Teh Manis,1.0,6000.0,6000.0
3,Paket Sop Iga,1.0,36000.0,36000.0
4,Koin Parkir Motor,1.0,2000.0,2000.0
5,SUBTOTAL,NaN,NaN,74210.0
6,Pembulatan,NaN,NaN,-10.0
7,TOTAL,NaN,NaN,74200.0


### Interpretasi dataset

Dataset terdiri dari **20 foto nota** dan setiap foto memiliki ground truth sendiri (bisa dilihaht pada folder research/ground_truth). Jumlah ini cukup untuk membandingkan perilaku ketiga pendekatan dalam scope mini project, tetapi belum cukup
untuk menyatakan performa umum pada semua jenis receipt di dunia nyata.

Ground truth tetap dianggap sebagai acuan. Prediction model **tidak digunakan untuk mengubah ground truth**, meskipun secara hitungan model terlihat berbeda.


## 2. Prompt untuk Model VLM Generatif (DeepSeek & Qwen)

`RECEIPT_PROMPT` digunakan oleh DeepSeek dan Qwen untuk mengarahkan model mengekstrak informasi nota ke dalam format JSON yang konsisten.

Donut tidak menggunakan prompt ini karena merupakan model OCR-free document understanding yang telah di-fine-tune khusus untuk memahami struktur dokumen/receipt.

In [7]:
print(RECEIPT_PROMPT)

Kamu adalah sistem pembaca struk belanja. Baca gambar struk ini dan kembalikan
HANYA JSON (tanpa penjelasan) dengan format:

{
  "merchant_name": "nama toko atau null",
  "items": [
    {"name": "nama item", "quantity": 1, "unit_price": 10000, "total_price": 10000}
  ],
  "subtotal": 10000,
  "charges": [
    {"name": "Pajak 10%", "amount": 1000}
  ],
  "total": 11000
}

Aturan:
- Semua harga ditulis sebagai angka polos tanpa "Rp", titik, atau koma ribuan.
  Contoh: "25.000" ditulis 25000.
- quantity = 1 jika jumlah tidak tertulis.
- total_price = total harga baris item (quantity x unit_price).
- subtotal = jumlah harga semua item sebelum pajak/service/diskon.
- charges berisi semua biaya di antara subtotal dan total: pajak/PB1/PPN, service
  charge, diskon, pembulatan, ongkir, dll. Diskon ditulis NEGATIF.
- Jangan masukkan baris pembayaran seperti Tunai/Cash, Kembali/Change, atau
  Debit/QRIS ke dalam items maupun charges.
- total = grand total yang harus dibayar.


### Alasan model Donut tidak memakai prompt yang sama

Karena DeepSeek dan Qwen adalah model generatif berbasis vision-language, sehingga keduanya menerima **gambar + instruksi** dan diminta menghasilkan JSON. Sedangkan Donut bekerja dengan berbeda, model ini sudah di-*fine-tune* untuk memahami dokumen/receipt, sehingga tidak menggunakan `RECEIPT_PROMPT` yang sama. Karena itu research ini sebenarnya membandingkan dua pendekatan:

- **Generative VLM:** DeepSeek dan Qwen
- **Document understanding khusus receipt:** Donut

Perbedaan pendekatan ini penting saat membaca hasil akhir, karena kemampuan menghasilkan JSON yang valid juga menjadi bagian dari reliability model.


## 3. Menjalankan Model

Setiap sel di bawah memuat satu model, membaca seluruh dataset nota, lalu menampilkan hasil extraction dan metrik evaluasinya.

Konfigurasi benchmark:

- **DeepSeek-V4.1-Flash**: 1 inference per nota (`DEEPSEEK_RUNS = 1`)
- **Donut CORD-v2**: 3 inference per nota (`RUNS = 3`)
- **Qwen3-VL-2B-Instruct**: 3 inference per nota (`RUNS = 3`)

DeepSeek hanya dijalankan 1x per nota karena menggunakan cloud API, sedangkan model lokal dijalankan tiga kali untuk memperoleh pengukuran
waktu inference yang lebih representatif.

Output prediction disimpan di: `research/results/predictions/`

In [8]:
all_runs, all_acc, models_info = [], [], []


def run_and_show(model_key):
    """Jalankan benchmark dan tampilkan hasil extraction satu model.

    DeepSeek menggunakan satu inference per nota untuk menghemat penggunaan API.
    Donut dan Qwen menggunakan jumlah run global untuk pengukuran latency lokal.
    """

    model_runs = (
        DEEPSEEK_RUNS
        if model_key == "deepseek"
        else RUNS
    )

    print(
        f"Menjalankan {model_key}: "
        f"{model_runs} run per nota"
    )

    runs, acc, info = benchmark_model(
        model_key,
        dataset,
        model_runs,
    )

    all_runs.extend(runs)
    all_acc.extend(acc)
    models_info.append(info)

    if "error" in info:
        print(
            "Model gagal dimuat:",
            info["error"],
        )
        return

    print(
        json.dumps(
            info,
            indent=2,
        )
    )

    for name, _, _ in dataset:
        pred = json.loads(
            (
                benchmark.RESULTS_DIR
                / "predictions"
                / model_key
                / f"{name}.json"
            ).read_text()
        )

        print(
            f"\n=== {name} ==="
        )

        if pred["parsed"] is None:
            print(
                "GAGAL:",
                pred["error"],
            )
            continue

        from modules.schema import Receipt

        display(
            receipt_table(
                Receipt.model_validate(
                    pred["parsed"]
                )
            )
        )

    display(
        pd.DataFrame(acc)
        .set_index("receipt")
        .T
    )

### 3.1 DeepSeek V4.1 Flash (API)

In [9]:
from modules.readers import create_reader

reader = create_reader("deepseek")

print(type(reader))
print(reader.label)

<class 'modules.readers.deepseek_reader.DeepSeekReader'>
DeepSeek (deepseek-flash)


In [10]:
run_and_show("deepseek")

Menjalankan deepseek: 1 run per nota
[DeepSeek (deepseek-flash)] load 0.0 detik di cloud API
  nota_01 run 1: 1.65 detik
  nota_02 run 1: 1.47 detik
  nota_03 run 1: 1.61 detik
  nota_04 run 1: 2.37 detik
  nota_05 run 1: 1.72 detik
  nota_06 run 1: 4.20 detik
  nota_07 run 1: 1.76 detik
  nota_08 run 1: 1.36 detik
  nota_09 run 1: 1.48 detik
  nota_10 run 1: 1.60 detik
  nota_11 run 1: 2.04 detik
  nota_12 run 1: 1.54 detik
  nota_13 run 1: 1.42 detik
  nota_14 run 1: 1.35 detik
  nota_15 run 1: 1.30 detik
  nota_16 run 1: 0.89 detik
  nota_17 run 1: 1.28 detik
  nota_18 run 1: 1.48 detik
  nota_19 run 1: 1.36 detik
  nota_20 run 1: 1.51 detik
{
  "model": "DeepSeek (deepseek-flash)",
  "key": "deepseek",
  "load_seconds": 0.0,
  "memory_mb_after_load": 0.0,
  "device": "cloud API"
}

=== nota_01 ===


,name,quantity,unit_price,total_price
0,SARI ROTI SW CK,1.0,4500.0,4500.0
1,SUBTOTAL,NaN,NaN,4500.0
2,PPN,NaN,NaN,500.0
3,TOTAL,NaN,NaN,5000.0



=== nota_02 ===


,name,quantity,unit_price,total_price
0,SARI ROTI KRIM CK,1.0,3500.0,3500.0
1,OVALTINE 3IN1 12X,1.0,28500.0,28500.0
2,SUBTOTAL,NaN,NaN,32000.0
3,PPN,NaN,NaN,2909.0
4,TOTAL,NaN,NaN,34909.0



=== nota_03 ===


...


=== nota_19 ===


,name,quantity,unit_price,total_price
0,Pistachoco,1.0,12127.0,12127.0
1,Kiwi Breeze,1.0,12127.0,12127.0
2,Air Mineral,1.0,8181.0,8181.0
3,SUBTOTAL,NaN,NaN,33635.0
4,PB1,NaN,NaN,3364.0
5,Pembulatan,NaN,NaN,1.0
6,TOTAL,NaN,NaN,37000.0



=== nota_20 ===


,name,quantity,unit_price,total_price
0,Ayam Bakar Rica,1.0,22210.0,22210.0
1,Dada,1.0,0.0,0.0
2,Paket B,1.0,8000.0,8000.0
3,Teh Manis,1.0,6000.0,6000.0
4,Dungin,1.0,0.0,0.0
5,Paket Sop Iga,1.0,36000.0,36000.0
6,Koin Parkir Motor,1.0,2000.0,2000.0
7,SUBTOTAL,NaN,NaN,74210.0
8,Diskon,NaN,NaN,0.0
9,Pembulatan,NaN,NaN,-10.0


receipt,nota_01,nota_02,nota_03,nota_04,nota_05,nota_06,nota_07,nota_08,nota_09,nota_10,nota_11,nota_12,nota_13,nota_14,nota_15,nota_16,nota_17,nota_18,nota_19,nota_20
model,DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash),DeepSeek (deepseek-flash)
parsed,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
n_gt_items,1,2,3,11,3,22,3,1,2,3,10,3,4,1,4,1,2,2,3,5
n_pred_items,1,2,3,11,3,22,3,6,2,4,9,6,4,2,4,1,2,2,3,7
item_precision,1.0,1.0,1.0,1.0,1.0,0.181818,1.0,0.166667,1.0,0.75,1.0,0.5,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.714286
item_recall,1.0,1.0,1.0,1.0,1.0,0.181818,1.0,1.0,1.0,1.0,0.9,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
item_f1,1.0,1.0,1.0,1.0,1.0,0.181818,1.0,0.285714,1.0,0.857143,0.947368,0.666667,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.833333
name_cer,0.0,0.03125,0.071429,0.013636,0.085145,0.575815,0.037885,0.0,0.0,0.060606,0.071902,0.0,0.257143,1.0,0.27381,0.0,0.1,0.566667,0.074074,0.028571
qty_acc,1.0,1.0,1.0,1.0,1.0,0.181818,1.0,1.0,1.0,1.0,0.8,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
unit_price_acc,1.0,1.0,1.0,1.0,1.0,0.045455,1.0,1.0,1.0,1.0,0.8,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.333333,1.0


#### Interpretasi DeepSeek

Pada run yang tersimpan di notebook ini:

- **20/20 inference berhasil diparse** → `run_success_rate = 100%`
- **20/20 nota berhasil** → `receipt_parse_rate = 100%`
- `item_f1 = 0.839`
- `name_cer = 0.162` → semakin kecil semakin baik
- `total_price_acc = 0.759`
- `subtotal_ok = 0.750`
- `total_ok = 0.850`
- rata-rata inference sekitar **1.67 detik/nota**

Secara keseluruhan DeepSeek memberikan hasil paling seimbang antara **akurasi, reliability, dan kecepatan**.
Structured output dari API juga membuat output lebih stabil untuk langsung masuk ke schema aplikasi.

Kekurangannya adalah model berjalan di **cloud**, sehingga membutuhkan koneksi internet, API key, dan ada pertimbangan biaya serta privasi data dibanding model lokal.

> **Note:** DeepSeek hanya diuji **1 kali per nota** untuk menghemat penggunaan API. Jadi reliability 100% pada model ini
> berasal dari 20 inference, sedangkan model lokal diuji 60 inference.


### 3.2 Donut CORD-v2 (lokal)

In [11]:
run_and_show("donut")

Menjalankan donut: 3 run per nota


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[Donut (CORD-v2)] load 6.6 detik di mps
  nota_01 run 1: 2.54 detik
  nota_01 run 2: 1.38 detik
  nota_01 run 3: 1.37 detik
  nota_02 run 1: 2.78 detik
  nota_02 run 2: 2.06 detik
  nota_02 run 3: 2.04 detik
  nota_03 run 1: 1.40 detik
  nota_03 run 2: 1.38 detik
  nota_03 run 3: 1.35 detik
  nota_04 run 1: 3.86 detik
  nota_04 run 2: 2.98 detik
  nota_04 run 3: 2.95 detik
  nota_05 run 1: 1.73 detik
  nota_05 run 2: 1.70 detik
  nota_05 run 3: 1.70 detik
  nota_06 run 1: 3.38 detik
  nota_06 run 2: 3.17 detik
  nota_06 run 3: 3.18 detik
  nota_07 run 1: 2.25 detik
  nota_07 run 2: 2.25 detik
  nota_07 run 3: 2.25 detik
  nota_08 run 1: 1.57 detik
  nota_08 run 2: 1.61 detik
  nota_08 run 3: 1.62 detik
  nota_09 run 1: 1.00 detik
  nota_09 run 2: 0.97 detik
  nota_09 run 3: 0.98 detik
  nota_10 run 1: 1.25 detik
  nota_10 run 2: 1.27 detik
  nota_10 run 3: 1.25 detik
  nota_11 run 1: 8.86 detik
  nota_11 run 2: 5.60 detik
  nota_11 run 3: 5.60 detik
  nota_12 run 1: 1.46 detik
  nota_1

,name,quantity,unit_price,total_price
0,ALFAMART STA.KARET,1.0,NaN,-1.336239e+13
1,SARI ROTI SW CK,1.0,4500.0,4.500000e+03
2,SUBTOTAL,NaN,NaN,4.500000e+03
3,Pajak,NaN,NaN,4.500000e+03
4,TOTAL,NaN,NaN,5.000000e+03



=== nota_02 ===


...


=== nota_19 ===


,name,quantity,unit_price,total_price
0,Hiro Donuts & Coffee Alianyang,1.0,NaN,1114.0
1,Jam Masuk Quick Service,12.0,-120920205.0,1114.0
2,Kasir KASIR Al IANYANG,1.0,12727.0,8181.0
3,SUBTOTAL,NaN,NaN,36999.0
4,Pajak,NaN,NaN,3364.0
5,TOTAL,NaN,NaN,37000.0



=== nota_20 ===


,name,quantity,unit_price,total_price
0,"Il.Metdeke No.01, Tengah, Kec. Pontanak Kota, K oto Pont...",1.0,NaN,78111.0
1,Jl. Merdeka Barat No. 1,628.0,6.281520e+11,78111.0
2,Dada,1.0,1.147000e+03,0.0
3,Paket B,1.0,NaN,8000.0
4,Teh Menis,1.0,NaN,6000.0
5,Drugin,1.0,NaN,0.0
6,Paket Sop Iga,1.0,NaN,36000.0
7,Kom Parkir Motor,1.0,NaN,2000.0
8,SUBTOTAL,NaN,NaN,74210.0
9,Lain-lain,NaN,NaN,-10.0


receipt,nota_01,nota_02,nota_03,nota_04,nota_05,nota_06,nota_07,nota_08,nota_09,nota_10,nota_11,nota_12,nota_13,nota_14,nota_15,nota_16,nota_17,nota_18,nota_19,nota_20
model,Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2),Donut (CORD-v2)
parsed,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
n_gt_items,1,2,3,11,3,22,3,1,2,3,10,3,4,1,4,1,2,2,3,5
n_pred_items,2,1,2,13,5,5,4,2,3,4,0,4,7,0,8,3,4,0,3,8
item_precision,0.5,1.0,0.5,0.846154,0.6,0.4,0.0,0.5,0.333333,0.75,0.0,0.75,0.571429,0.0,0.5,0.333333,0.0,0.0,0.333333,0.5
item_recall,1.0,0.5,0.333333,1.0,1.0,0.090909,0.0,1.0,0.5,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.333333,0.8
item_f1,0.666667,0.666667,0.4,0.916667,0.75,0.148148,0.0,0.666667,0.4,0.857143,0.0,0.857143,0.727273,0.0,0.666667,0.5,0.0,0.0,0.333333,0.615385
name_cer,0.0,0.125,0.076923,0.031818,0.342852,0.630952,1.0,0.0,0.761905,0.17316,1.0,0.457576,0.020833,1.0,0.083333,0.2,1.0,1.0,1.363636,0.092904
qty_acc,1.0,0.5,0.333333,1.0,1.0,0.090909,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.333333,0.8
unit_price_acc,1.0,0.0,0.0,0.909091,0.666667,0.0,0.0,0.0,0.0,0.333333,0.0,0.333333,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


#### Interpretasi Donut

Pada notebook ini Donut memiliki:

- **60/60 inference berhasil diparse** → `run_success_rate = 100%`
- **20/20 nota berhasil diparse** → `receipt_parse_rate = 100%`
- rata-rata inference sekitar **2.15 detik/nota**
- `item_f1 = 0.459`
- `name_cer = 0.468`
- `unit_price_acc = 0.212`
- `total_price_acc = 0.382`
- `total_ok = 0.350`
- `consistent = 0.000`

Artinya Donut **stabil secara teknis** karena hampir selalu menghasilkan output yang bisa diproses, tetapi isi hasil ekstraksinya masih jauh di bawah DeepSeek dan Qwen pada dataset ini. Hal ini masuk akal karena Donut CORD-v2 adalah model yang di-*fine-tune* pada domain/dataset tertentu.
Format nota penelitian ini tidak selalu sama dengan pola data yang pernah dipelajari model.

> Kesimpulan: **Donut bagus sebagai baseline model khusus dokumen, tetapi belum menjadi pilihan utama untuk ekstraksi SmartSplit Bill pada dataset ini.**


### 3.3 Qwen3-VL-2B-Instruct (lokal)

In [12]:
run_and_show("qwen")

Menjalankan qwen: 3 run per nota
[Qwen-VL (Qwen3-VL-2B-Instruct)] load 10.8 detik di mps
  nota_01 run 1: 8.31 detik
  nota_01 run 2: 5.15 detik
  nota_01 run 3: 5.85 detik
  nota_02 run 1: 7.74 detik
  nota_02 run 2: 7.18 detik
  nota_02 run 3: 7.32 detik
  nota_03 run 1: 9.14 detik
  nota_03 run 2: 8.47 detik
  nota_03 run 3: 8.71 detik
  nota_04 run 1: 28.03 detik
  nota_04 run 2: 22.50 detik
  nota_04 run 3: 22.34 detik
  nota_05 run 1: 9.63 detik
  nota_05 run 2: 9.48 detik
  nota_05 run 3: 8.95 detik
  nota_06 run 1: 23.73 detik
  nota_06 run 2: 34.05 detik
  nota_06 run 3: ERROR Output Qwen bukan JSON yang valid.
Parser error: Expecting ',' delimiter: line 117 column 6 (char 2415)
Generated tokens
  nota_07 run 1: ERROR Output Qwen bukan JSON yang valid.
Parser error: Expecting ',' delimiter: line 29 column 21 (char 623)
Generated tokens:
  nota_07 run 2: 9.25 detik
  nota_07 run 3: 9.25 detik
  nota_08 run 1: 10.16 detik
  nota_08 run 2: 10.01 detik
  nota_08 run 3: 11.39 detik

,name,quantity,unit_price,total_price
0,SARI ROTI SW CK,1.0,4500.0,4500.0
1,SUBTOTAL,NaN,NaN,4500.0
2,PPN,NaN,NaN,500.0
3,TOTAL,NaN,NaN,5000.0



=== nota_02 ===


,name,quantity,unit_price,total_price
0,SARI ROTI KREM CK,1.0,3500.0,3500.0
1,OVALTINE 3IN1 12X,1.0,28500.0,28500.0
2,SUBTOTAL,NaN,NaN,32000.0
3,PPN,NaN,NaN,2909.0
4,TOTAL,NaN,NaN,34909.0



=== nota_03 ===


...


=== nota_19 ===


,name,quantity,unit_price,total_price
0,Pistachio,1.0,12727.0,12727.0
1,Kiwi Breeze,1.0,12727.0,12727.0
2,Air Mineral,1.0,6181.0,6181.0
3,SUBTOTAL,NaN,NaN,33635.0
4,PB1,NaN,NaN,3364.0
5,TOTAL,NaN,NaN,36999.0



=== nota_20 ===


,name,quantity,unit_price,total_price
0,Ayam Bakar Rica,1.0,22100.0,22100.0
1,Paket B,1.0,8000.0,8000.0
2,Teh Manis,1.0,6000.0,6000.0
3,Paket Sop Iga,1.0,36000.0,36000.0
4,Koin Parkir Motor,1.0,2000.0,2000.0
5,SUBTOTAL,NaN,NaN,74210.0
6,Pembulatan,NaN,NaN,-10.0
7,TOTAL,NaN,NaN,74200.0


receipt,nota_01,nota_02,nota_03,nota_04,nota_05,nota_06,nota_07,nota_08,nota_09,nota_10,nota_11,nota_12,nota_13,nota_14,nota_15,nota_16,nota_17,nota_18,nota_19,nota_20
model,Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct),Qwen-VL (Qwen3-VL-2B-Instruct)
parsed,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
n_gt_items,1,2,3,11,3,22,3,1,2,3,10,3,4,1,4,1,2,2,3,5
n_pred_items,1,2,3,11,3,11,3,5,2,3,9,2,4,2,4,1,2,4,3,5
item_precision,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.5,1.0,1.0,1.0,0.25,1.0,1.0
item_recall,1.0,1.0,1.0,1.0,1.0,0.5,1.0,0.0,1.0,0.0,0.9,0.666667,1.0,1.0,1.0,1.0,1.0,0.5,1.0,1.0
item_f1,1.0,1.0,1.0,1.0,1.0,0.666667,1.0,0.0,1.0,0.0,0.947368,0.8,1.0,0.666667,1.0,1.0,1.0,0.333333,1.0,1.0
name_cer,0.0,0.03125,0.071429,0.022727,0.0,0.151287,0.007246,1.0,0.0,1.0,0.013889,0.05,0.0,0.636364,0.0,0.0,0.12,0.266667,0.0,0.028571
qty_acc,1.0,1.0,1.0,1.0,1.0,0.454545,1.0,0.0,1.0,0.0,0.9,0.666667,1.0,1.0,1.0,1.0,1.0,0.5,1.0,1.0
unit_price_acc,1.0,1.0,0.666667,1.0,1.0,0.0,0.666667,0.0,1.0,0.0,0.8,0.666667,1.0,1.0,1.0,1.0,0.5,0.0,0.666667,0.8


#### Interpretasi Qwen dan eksperimen structured output

Qwen memberi hasil yang menarik karena akurasi item-nya mendekati DeepSeek, tetapi reliability output nya tidak selalu stabil.

Pada run yang **tersimpan di notebook ini**:

- total inference = **60**
- berhasil = **57**
- gagal parse = **3**
- `run_success_rate = 95%`
- semua **20/20 nota berhasil minimal sekali** → `receipt_parse_rate = 100%`
- run yang gagal pada hasil ini terjadi pada **nota_06 run 3, nota_07 run 1, dan nota_14 run 3**
- `item_f1 = 0.821`
- `name_cer = 0.170`
- `total_price_acc = 0.688`
- `total_ok = 0.300`
- rata-rata inference sekitar **10.70 detik/nota**

##### Temuan penting dari eksperimen Qwen

Pada konfigurasi awal dengan **greedy decoding (`do_sample=False`)**, `nota_14` gagal berulang kali. Debugging menunjukkan model menghasilkan tepat **1024 token**, mencapai `MAX_NEW_TOKENS`, lalu bagian akhir output berubah menjadi pengulangan token seperti `1 1 1 1 ...`.

Artinya masalahnya bukan MPS crash dan bukan parser yang salah. Model mengalami **degenerate repetition** sehingga JSON tidak selesai dengan benar.

Kemudian generation Qwen diubah mengikuti sampling model:

- `do_sample=True`
- `temperature=0.7`
- `top_p=0.8`
- `top_k=20`

Hasilnya lebih baik karena `nota_14` dapat berhasil pada sebagian run. Namun hasil antar-run menjadi **tidak deterministik** karena receipt yang sama bisa berhasil pada satu run dan gagal pada run lain.

Ini menunjukkan keterbatasan penting model lokal 2B yaitu **prompt saja belum menjamin structured output selalu valid**. Qwen tetap menarik karena berjalan lokal dan `item_f1` cukup tinggi, tetapi aplikasi perlu memiliki validation dan fallback ketika JSON gagal.

## 4. Perbandingan

In [13]:
summary = save_results(all_runs, all_acc, models_info)
summary.set_index("model").T

model,DeepSeek (deepseek-flash),Donut (CORD-v2),Qwen-VL (Qwen3-VL-2B-Instruct)
key,deepseek,donut,qwen
load_seconds,0.0,6.63,10.83
memory_mb_after_load,0.0,480.8,-1841.1
device,cloud API,mps,mps
first_run_s,1.65,2.539,8.315
mean_s,1.67,2.149,10.698
min_s,0.892,0.212,5.154
max_s,4.202,8.864,34.047
total_runs,20,60,60
successful_runs,20,60,57


> **Catatan memori:** nilai `memory_mb_after_load` yang negatif pada Qwen tidak berarti Qwen memakai RAM negatif.
> Nilai tersebut berasal dari selisih RSS proses sebelum dan sesudah load model. Pada PyTorch/MPS, allocator,
> garbage collection, dan pelepasan memori model sebelumnya dapat membuat delta RSS turun. Karena itu angka ini
> tidak dipakai sebagai dasar utama memilih model.


In [14]:
# Ringkasan reliability setiap model
reliability_columns = [
    "model",
    "total_runs",
    "successful_runs",
    "failed_runs",
    "run_success_rate",
    "total_receipts",
    "parsed_receipts",
    "failed_receipts",
    "receipt_parse_rate",
]

reliability_summary = summary[
    reliability_columns
].copy()

reliability_summary[
    "run_success_rate"
] = (
    reliability_summary[
        "run_success_rate"
    ]
    * 100
).round(1)

reliability_summary[
    "receipt_parse_rate"
] = (
    reliability_summary[
        "receipt_parse_rate"
    ]
    * 100
).round(1)

reliability_summary = (
    reliability_summary.rename(
        columns={
            "run_success_rate":
                "run_success_rate_%",
            "receipt_parse_rate":
                "receipt_parse_rate_%",
        }
    )
)

display(
    reliability_summary
)

,model,total_runs,successful_runs,failed_runs,run_success_rate_%,total_receipts,parsed_receipts,failed_receipts,receipt_parse_rate_%
0,DeepSeek (deepseek-flash),20,20,0,100.0,20,20,0,100.0
1,Donut (CORD-v2),60,60,0,100.0,20,20,0,100.0
2,Qwen-VL (Qwen3-VL-2B-Instruct),60,57,3,95.0,20,20,0,100.0


### Interpretasi Reliability

Dua angka reliability sengaja dipisahkan:

- **Run success rate** menjawab: *dari semua percobaan inference, berapa persen yang langsung usable?*
- **Receipt parse rate** menjawab: *dari seluruh jenis nota, berapa persen yang pernah berhasil minimal satu kali?*

Pada Qwen, `receipt_parse_rate = 100%` **tidak berarti model selalu berhasil**. Model tetap memiliki 3 run gagal, sehingga `run_success_rate = 95%`.

Pemisahan ini penting karena accuracy notebook dihitung dari **first successful result**. Tanpa reliability metric, model yang gagal beberapa kali tetapi akhirnya berhasil bisa terlihat lebih stabil daripada kondisi sebenarnya.

In [15]:
runs_df = pd.DataFrame(all_runs).dropna(subset=["seconds"])
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

speed = summary.set_index("model")[["load_seconds", "mean_s"]]
speed.plot.barh(ax=axes[0], logx=True)
axes[0].set_title("Waktu load model & rata-rata inference per nota (detik, skala log)")
axes[0].set_xlabel("detik")

metric_cols = ["item_f1", "total_price_acc", "subtotal_ok", "charges_recall", "total_ok", "consistent"]
summary.set_index("model")[metric_cols].T.plot.bar(ax=axes[1], rot=30)
axes[1].set_title("Akurasi per field (1.0 = sempurna)")
axes[1].set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

/var/folders/7j/1_v4mdn15nn4ry80ckw0xy9w0000gn/T/ipykernel_11235/608485870.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# waktu tiap run: run pertama biasanya lebih lambat (warm-up / cache)
runs_df.pivot_table(index=["model", "receipt"], columns="run", values="seconds").round(2)

run                                         1      2      3
model                          receipt                     
DeepSeek (deepseek-flash)      nota_01   1.65    NaN    NaN
                               nota_02   1.47    NaN    NaN
                               nota_03   1.61    NaN    NaN
                               nota_04   2.37    NaN    NaN
                               nota_05   1.72    NaN    NaN
                               nota_06   4.20    NaN    NaN
                               nota_07   1.76    NaN    NaN
                               nota_08   1.36    NaN    NaN
                               nota_09   1.48    NaN    NaN
                               nota_10   1.60    NaN    NaN
                               nota_11   2.04    NaN    NaN
                               nota_12   1.54    NaN    NaN
                               nota_13   1.42    NaN    NaN
                               nota_14   1.35    NaN    NaN
                               nota_15   1.30    NaN    NaN
                               nota_16   0.89    NaN    NaN
                               nota_17   1.28    NaN    NaN
                               nota_18   1.48    NaN    NaN
                               nota_19   1.36    NaN    NaN
                               nota_20   1.51    NaN    NaN
Donut (CORD-v2)                nota_01   2.54   1.38   1.37
                               nota_02   2.78   2.06   2.04
                               nota_03   1.40   1.38   1.35
                               nota_04   3.86   2.98   2.95
                               nota_05   1.73   1.70   1.70
                               nota_06   3.38   3.17   3.18
                               nota_07   2.25   2.25   2.25
                               nota_08   1.57   1.61   1.62
                               nota_09   1.00   0.97   0.98
                               nota_10   1.25   1.27   1.25
                               nota_11   8.86   5.60   5.60
                               nota_12   1.46   1.40   1.41
                               nota_13   1.96   1.93   1.93
                               nota_14   0.24   0.22   0.21
                               nota_15   1.90   1.97   1.92
                               nota_16   1.35   1.35   1.38
                               nota_17   1.03   1.03   1.03
                               nota_18   5.58   5.59   5.55
                               nota_19   1.32   1.30   1.30
                               nota_20   1.75   1.76   1.77
Qwen-VL (Qwen3-VL-2B-Instruct) nota_01   8.31   5.15   5.85
                               nota_02   7.74   7.18   7.32
                               nota_03   9.14   8.47   8.71
                               nota_04  28.03  22.50  22.34
                               nota_05   9.63   9.48   8.95
                               nota_06  23.73  34.05    NaN
                               nota_07    NaN   9.25   9.25
                               nota_08  10.16  10.01  11.39
                               nota_09   5.61   6.72   6.71
                               nota_10   9.29   9.19   9.29
                               nota_11  18.92  19.11  19.11
                               nota_12   7.44   8.85   8.89
                               nota_13  10.64  11.08  10.77
                               nota_14   6.96   7.01    NaN
                               nota_15  10.65  10.75  10.60
                               nota_16   5.66   5.61   5.53
                               nota_17   7.40   7.46   7.47
                               nota_18  10.03   8.66   5.76
                               nota_19   8.61   8.57   8.60
                               nota_20  12.51  11.80  11.88

### Interpretasi kecepatan

Rata-rata inference pada hasil notebook ini:

| Model | Rata-rata inference | Catatan |
|---|---:|---|
| DeepSeek | **1.67 s** | Cloud API, 1 run per nota |
| Donut | **2.15 s** | Lokal MPS |
| Qwen3-VL-2B | **10.70 s** | Lokal MPS |

DeepSeek terlihat paling cepat pada eksperimen ini, tetapi perbandingannya perlu dibaca dengan konteks. Komputasi utama DeepSeek terjadi di server cloud, sedangkan Donut dan Qwen menghitung langsung di MacBook.

Untuk pilihan **lokal**, Donut jauh lebih cepat daripada Qwen, tetapi akurasinya lebih rendah. Qwen membutuhkan waktu lebih lama karena melakukan generative vision-language inference.


In [17]:
# perbandingan nama item yang terbaca tiap model, per nota
for name, _, gt in dataset:
    columns = {
        "ground_truth": [
            f"{i.name} | {i.quantity:g} | {i.total_price:,.0f}"
            for i in gt.items
        ]
    }

    for key in [
        "deepseek",
        "donut",
        "qwen",
    ]:
        path = (
            benchmark.RESULTS_DIR
            / "predictions"
            / key
            / f"{name}.json"
        )

        if path.exists():
            parsed = (
                json.loads(
                    path.read_text()
                )["parsed"]
                or {"items": []}
            )

            columns[key] = [
                (
                    f"{i['name']} | "
                    f"{i['quantity']:g} | "
                    f"{i['total_price']:,.0f}"
                )
                for i in parsed["items"]
            ]

    print(
        f"=== {name} ==="
    )

    display(
        pd.DataFrame(
            {
                key: pd.Series(values)
                for key, values in columns.items()
            }
        ).fillna("")
    )

=== nota_01 ===


,ground_truth,deepseek,donut,qwen
0,"SARI ROTI SW CK | 1 | 4,500","SARI ROTI SW CK | 1 | 4,500","ALFAMART STA.KARET | 1 | -13,362,389,054,000","SARI ROTI SW CK | 1 | 4,500"
1,,,"SARI ROTI SW CK | 1 | 4,500",


=== nota_02 ===


,ground_truth,deepseek,donut,qwen
0,"SARI ROTI KRM CK | 1 | 3,500","SARI ROTI KRIM CK | 1 | 3,500","SARI ROTI KRIN CK | 1 | 28,500","SARI ROTI KREM CK | 1 | 3,500"
1,"OVALTINE 3IN1 12X | 1 | 28,500","OVALTINE 3IN1 12X | 1 | 28,500",,"OVALTINE 3IN1 12X | 1 | 28,500"


=== nota_03 ===


...

,ground_truth,deepseek,donut,qwen
0,"9PCS CHIC-WINGS | 2 | 304,546","2 PCS Chick-Inner | 2 | 60,909",,"2 Pcs Chic-King | 2 | 33,000"
1,"CHAFEE TA | 4 | 7,272","4 CHICKEN TA | 4 | 250,909",,"1 PC Sundae | 1 | 20,000"
2,,,,"1 Drink | 1 | 5,000"
3,,,,"1 Drink | 1 | 5,000"


=== nota_19 ===


,ground_truth,deepseek,donut,qwen
0,"Pistachio | 1 | 12,727","Pistachoco | 1 | 12,127","Hiro Donuts & Coffee Alianyang | 1 | 1,114","Pistachio | 1 | 12,727"
1,"Kiwi Breeze | 1 | 12,727","Kiwi Breeze | 1 | 12,127","Jam Masuk Quick Service | 12 | 1,114","Kiwi Breeze | 1 | 12,727"
2,"Air Mineral | 1 | 8,181","Air Mineral | 1 | 8,181","Kasir KASIR Al IANYANG | 1 | 8,181","Air Mineral | 1 | 6,181"


=== nota_20 ===


,ground_truth,deepseek,donut,qwen
0,"Ayam Bakar Rica | 1 | 22,210","Ayam Bakar Rica | 1 | 22,210","Il.Metdeke No.01, Tengah, Kec. Pontanak Kota, K oto Pont...","Ayam Bakar Rica | 1 | 22,100"
1,"Paket R | 1 | 8,000",Dada | 1 | 0,"Jl. Merdeka Barat No. 1 | 628 | 78,111","Paket B | 1 | 8,000"
2,"Teh Manis | 1 | 6,000","Paket B | 1 | 8,000",Dada | 1 | 0,"Teh Manis | 1 | 6,000"
3,"Paket Sop Iga | 1 | 36,000","Teh Manis | 1 | 6,000","Paket B | 1 | 8,000","Paket Sop Iga | 1 | 36,000"
4,"Koin Parkir Motor | 1 | 2,000",Dungin | 1 | 0,"Teh Menis | 1 | 6,000","Koin Parkir Motor | 1 | 2,000"
5,,"Paket Sop Iga | 1 | 36,000",Drugin | 1 | 0,
6,,"Koin Parkir Motor | 1 | 2,000","Paket Sop Iga | 1 | 36,000",
7,,,"Kom Parkir Motor | 1 | 2,000",


### Interpretasi Perbandingan Setiap Model dengan Ground Truth per Nota

Perbandingan per-nota menunjukkan bahwa nilai rata-rata saja belum cukup. Model bisa mendapatkan skor keseluruhan cukup baik tetapi tetap salah pada receipt tertentu, misalnya:

- nama item sedikit berubah,
- item tambahan terbaca sebagai produk,
- harga satuan dan total tertukar,
- subtotal/total tidak konsisten,
- atau output JSON tidak selesai.

Karena hasil akhirnya dipakai untuk **split bill**, kesalahan angka lebih berbahaya daripada sekadar typo nama item. Itulah sebabnya metrik `unit_price_acc`, `total_price_acc`, `total_ok`, dan consistency tetap diperiksa, bukan hanya `item_f1`.

## 5. Analisis & Pemilihan Model

### 5.1 Ringkasan hasil utama

| Metrik | DeepSeek | Donut | Qwen3-VL-2B |
|---|---:|---:|---:|
| Item F1 | **0.839** | 0.459 | 0.821 |
| Name CER ↓ | **0.162** | 0.468 | 0.170 |
| Quantity accuracy | **0.899** | 0.553 | 0.826 |
| Unit price accuracy | **0.759** | 0.212 | 0.688 |
| Total price accuracy | **0.759** | 0.382 | 0.688 |
| Subtotal OK | **0.750** | 0.600 | 0.500 |
| Charges recall | **0.850** | 0.575 | 0.550 |
| Total OK | **0.850** | 0.350 | 0.300 |
| Consistent | **0.600** | 0.000 | **0.600** |
| Run success rate | **100%** | **100%** | 95% |
| Receipt parse rate | **100%** | **100%** | **100%** |
| Mean inference | **1.67 s** | 2.15 s | 10.70 s |


> Nilai di atas berasal dari hasil yang tersimpan pada notebook ini.
> DeepSeek dijalankan 1 kali per nota, sedangkan Donut dan Qwen 3 kali per nota.

### 5.2 Interpretasi sederhana

**DeepSeek** menjadi model terbaik secara keseluruhan pada eksperimen ini.
Nilai item F1 paling tinggi, kesalahan nama paling rendah, pembacaan total paling baik, semua output berhasil diparse,
dan waktu inference juga paling cepat pada setup ini.

**Qwen3-VL-2B** adalah alternatif lokal yang paling menarik.
Akurasi item-nya dekat dengan DeepSeek dan seluruh receipt berhasil dibaca minimal satu kali.
Kelemahannya adalah inference jauh lebih lambat dan structured output belum konsisten pada setiap run.
Eksperimen `nota_14` menunjukkan model bahkan dapat masuk repetition loop sampai menyentuh batas output token.

**Donut** adalah model lokal yang paling stabil dari sisi parsing dan lebih cepat daripada Qwen.
Namun akurasi item, harga, dan total masih rendah pada dataset ini. Karena itu Donut lebih cocok sebagai baseline
research daripada model utama aplikasi.

### 5.3 Model yang dipilih

Untuk **SmartSplit Bill**, pilihan utama dari hasil research ini adalah:

**1. DeepSeek-V4.1-Flash (model utama)**
- hasil paling seimbang,
- structured output paling stabil,
- pembacaan item dan total paling baik,
- inference cepat pada eksperimen ini.

**2. Qwen3-VL-2B-Instruct (alternatif lokal / fallback)**
- data dapat diproses sepenuhnya di komputer sendiri,
- akurasi item cukup dekat dengan DeepSeek,
- tetapi perlu validation karena JSON dapat gagal pada sebagian run.

**3. Donut CORD-v2 (baseline pembanding)**
- parsing stabil,
- lokal dan relatif cepat,
- tetapi akurasi belum cukup untuk menjadi reader utama pada dataset ini.

### 5.4 Temuan research yang paling penting

Research ini menunjukkan bahwa **akurasi bukan satu-satunya hal yang perlu dinilai**.

Untuk aplikasi receipt reader, model juga harus:
1. menghasilkan format yang dapat diproses,
2. stabil ketika receipt yang sama dibaca kembali,
3. membaca angka dengan benar,
4. menjaga subtotal dan total tetap masuk akal,
5. memiliki latency yang masih nyaman dipakai.

Karena itu benchmark final memakai tiga kelompok penilaian:

- **Accuracy** → seberapa benar isi extraction.
- **Reliability** → seberapa sering output benar-benar usable.
- **Latency** → seberapa cepat hasil diperoleh.

### 5.5 Keterbatasan eksperimen

Hasil ini tetap memiliki beberapa batasan:

- dataset hanya **20 nota**, jadi belum mewakili semua jenis receipt
- DeepSeek hanya **1 run per nota**, sedangkan model lokal 3 run
- Qwen menggunakan sampling sehingga hasil antar-run dapat berbeda
- DeepSeek berjalan di cloud sedangkan Qwen/Donut lokal, sehingga perbandingan resource tidak sepenuhnya apple-to-apple
- delta RSS pada MPS tidak cukup stabil untuk dijadikan ukuran utama penggunaan memori
- accuracy menggunakan **first successful result**, sehingga reliability harus dibaca bersama accuracy

### 5.6 Kesimpulan akhir

Dari eksperimen ini, **DeepSeek-V4.1-Flash dipilih sebagai reader utama SmartSplit Bill** karena memberikan
kombinasi terbaik antara akurasi, reliability, dan kecepatan.

**Qwen3-VL-2B-Instruct tetap layak dipertahankan sebagai opsi lokal**, terutama ketika privasi dan penggunaan
tanpa cloud lebih penting. Namun output harus tetap melewati validation karena structured JSON belum selalu stabil.

**Donut CORD-v2 tetap berguna sebagai baseline model document understanding**, tetapi pada dataset ini performa
ekstraksinya belum cukup kuat dibanding dua VLM.

Jadi keputusan akhir bukan sekadar memilih model dengan satu skor tertinggi, tetapi memilih model yang paling
sesuai dengan kebutuhan aplikasi seperti **hasil harus cukup akurat, bisa diproses, stabil, dan cepat.**
